In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import time
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Ensure data_utils is accessible
sys.path.append(os.path.abspath('.'))
import data_utils

plt.rc('text', usetex=False)
plt.rc('font', family='serif', size=12)

# Create output directories
os.makedirs('tsne_data', exist_ok=True)
os.makedirs('figures', exist_ok=True)


## Mathematical Framework: Manifold Learning across Runs

To analyze the underlying manifold of the turbulent jet at different actuation frequencies, we apply **t-Distributed Stochastic Neighbor Embedding (t-SNE)**. t-SNE is a nonlinear dimensionality reduction technique well-suited for embedding high-dimensional velocity snapshots into a 2D space while preserving local neighborhood structures (identifying clusters corresponding to distinct phase-locked vortex structures).

Because the raw spatial dimension of each snapshot is $M = 2 \times N_x \times N_y = 171,622$, computing pairwise distances for t-SNE directly is computationally prohibitive and sensitive to noise. Therefore, the computation is performed in two stages:
1. **Denoising via PCA**: We project the snapshot matrix $\mathbf{U} \in \mathbb{R}^{N_t \times M}$ onto its first 50 Principal Components. This captures the vast majority of the turbulent kinetic energy while dramatically reducing the feature space to $\mathbb{R}^{N_t \times 50}$.
2. **Nonlinear Embedding**: We apply t-SNE to the PCA scores with a perplexity of 30, transforming the data into the final manifold coordinates $\mathbf{Y} \in \mathbb{R}^{N_t \times 2}$.

This notebook iterates over all runs, computes the embedding, and caches the results in the `tsne_data/` directory.


In [ ]:
# Hyperparameters
n_pca_components = 50
perplexity = 30
random_state = 42

runs_to_process = range(1, 26)
total_runs = len(runs_to_process)

print(f"Starting t-SNE pipeline for {total_runs} runs...")

for run_idx in runs_to_process:
    output_path = f"tsne_data/RUN{run_idx}_tsne.npz"
    
    # Skip if already computed
    if os.path.exists(output_path):
        print(f"RUN {run_idx}: t-SNE already computed. Skipping.")
        continue
        
    data_path = f"compressed_data/RUN{run_idx}_PIV_compressed.npz"
    if not os.path.exists(data_path):
        print(f"RUN {run_idx}: Data file not found ({data_path}). Skipping.")
        continue
        
    print(f"\n--- Processing RUN {run_idx} ---")
    start_time = time.time()
    
    # 1. Load data
    print("  Loading compressed velocity data...")
    d = np.load(data_path)
    u = d['u']
    v = d['v']
    nt = u.shape[0]
    
    # 2. Flatten spatial dimensions
    print("  Flattening spatial dimensions...")
    u_flat = u.reshape(nt, -1)
    v_flat = v.reshape(nt, -1)
    X_snap = np.concatenate((u_flat, v_flat), axis=1)
    
    # Clear memory
    del u, v, u_flat, v_flat, d
    
    # 3. PCA
    print(f"  Computing PCA (n_components={n_pca_components})...")
    pca = PCA(n_components=n_pca_components, svd_solver='randomized', random_state=random_state)
    X_pca = pca.fit_transform(X_snap)
    var_explained = np.sum(pca.explained_variance_ratio_)
    print(f"    Variance explained by top {n_pca_components} PCs: {var_explained:.2%}")
    
    # Clear memory
    del X_snap
    
    # 4. t-SNE
    print(f"  Computing t-SNE (perplexity={perplexity})...")
    tsne = TSNE(n_components=2, perplexity=perplexity, init='pca', 
                learning_rate='auto', random_state=random_state, n_jobs=-1)
    X_tsne = tsne.fit_transform(X_pca)
    
    # 5. Save results
    computation_time = time.time() - start_time
    print(f"  Saving results to {output_path} (Took {computation_time:.1f} seconds)")
    
    np.savez_compressed(
        output_path,
        embedding=X_tsne,
        perplexity=perplexity,
        n_pca_components=n_pca_components,
        var_explained=var_explained,
        time_sec=computation_time
    )
    
print("\nAll runs processed successfully!")


## Visualization of the Learned Manifolds

After computation, we can visualize the 2D t-SNE embeddings for a few representative runs to observe how the manifold topology changes with different actuation frequencies.


In [ ]:
# Visualize a few representative runs
runs_to_plot = [1, 2, 4, 7, 10, 13] # Select some specific runs (Run 1 is unforced!)

fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=False, sharey=False)
axes = axes.flatten()

for i, run_idx in enumerate(runs_to_plot):
    if i >= len(axes): break
    ax = axes[i]
    
    output_path = f"tsne_data/RUN{run_idx}_tsne.npz"
    if not os.path.exists(output_path):
        ax.text(0.5, 0.5, f"RUN {run_idx}\nData not found", ha='center', va='center')
        ax.axis('off')
        continue
        
    data = np.load(output_path)
    emb = data['embedding']
    
    # Plot as a scatter plot
    ax.scatter(emb[:, 0], emb[:, 1], alpha=0.5, s=5, c='k')
    
    # Label
    run_label = "Unforced" if run_idx == 1 else f"Forced"
    ax.set_title(f"RUN {run_idx} ({run_label})", fontsize=14)
    ax.set_aspect('equal', 'datalim')
    ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/tsne_manifolds_overview.png', dpi=300, bbox_inches='tight')
plt.show()
